# SOLID Principles — Senior Backend Interview Guide

---

## 1. Simple Intuition First

Imagine you joined a 2-year-old startup. The codebase has a single `OrderService` class — it handles pricing, sends emails, charges payments, talks to inventory, and logs everything. It's 1500 lines long.

One day you need to add UPI support. You touch 5 methods. 3 bugs appear in email and inventory. Sound familiar?

**That's the problem SOLID solves.**

SOLID is a set of **5 design principles** that tell you how to structure your classes and modules so that:
- Changes in one part don't break others
- New features don't require rewriting existing code
- Code is testable without spinning up the entire system

Think of SOLID as **rules for writing code that doesn't rot** as your system grows.

---

## 2. Why This Topic Exists

### Real-world pain it solves:

| Symptom | Root Cause | SOLID Fix |
|---|---|---|
| Changing payment logic breaks email sending | God class / tight coupling | SRP + DIP |
| Adding new notification type requires `if-else` chains | Open-closed violation | OCP |
| Mock one DB, everything breaks in tests | DIP violation | DIP |
| Subclass breaks when parent is replaced | LSP violation | LSP |
| Forced to implement irrelevant interface methods | Fat interfaces | ISP |

### Why companies care in production:
- At **PhonePe/Razorpay**: payment modules change every sprint. If SRP/OCP is violated, every change is a fire drill.
- At **Zomato/Swiggy**: multiple notification channels (push, SMS, email, WhatsApp). Without OCP/ISP, adding WhatsApp means touching 12 files.
- At **Google/Uber**: microservices rely on DIP — services depend on contracts (interfaces), not concrete implementations. This is how you swap Kafka for Pub/Sub without breaking consumers.

---

## 3. Core Concepts

### S — Single Responsibility Principle (SRP)
> **One class = One reason to change**

A class should do one job and do it well. Not "one method", not "one line" — **one cohesive responsibility**.

```python
# BAD — God class
class OrderService:
    def create_order(self): ...
    def send_confirmation_email(self): ...   # email responsibility
    def charge_payment(self): ...            # payment responsibility
    def update_inventory(self): ...          # inventory responsibility
    def log_order(self): ...                 # logging responsibility

# GOOD — Each class owns one responsibility
class OrderService:
    def create_order(self): ...

class EmailNotifier:
    def send_confirmation(self, order): ...

class PaymentProcessor:
    def charge(self, order): ...
```

---

### O — Open/Closed Principle (OCP)
> **Open for extension, closed for modification**

You should be able to add new behavior **without touching existing tested code**.

```python
# BAD — Every new discount type = modify existing code = re-test everything
class DiscountCalculator:
    def calculate(self, order, discount_type):
        if discount_type == "festive":
            return order.total * 0.10
        elif discount_type == "loyalty":
            return order.total * 0.15
        elif discount_type == "new_user":   # added later — modified class
            return order.total * 0.20

# GOOD — Add new discount by adding a new class, not modifying existing
class Discount(ABC):
    @abstractmethod
    def calculate(self, order) -> float: ...

class FestiveDiscount(Discount):
    def calculate(self, order): return order.total * 0.10

class LoyaltyDiscount(Discount):
    def calculate(self, order): return order.total * 0.15

class NewUserDiscount(Discount):          # new feature = new class only
    def calculate(self, order): return order.total * 0.20
```

---

### L — Liskov Substitution Principle (LSP)
> **Subclasses must be replaceable for their base class without breaking behavior**

If class B extends A, then everywhere you use A, you should be able to drop in B without anything breaking.

```python
# BAD — Square breaks Rectangle's contract
class Rectangle:
    def set_width(self, w): self.width = w
    def set_height(self, h): self.height = h
    def area(self): return self.width * self.height

class Square(Rectangle):          # Square IS-A Rectangle? No!
    def set_width(self, w):
        self.width = w
        self.height = w           # silently changes height — breaks caller!

# GOOD — Don't force inheritance where substitution is invalid
class Shape(ABC):
    @abstractmethod
    def area(self): ...

class Rectangle(Shape): ...
class Square(Shape): ...          # separate hierarchy
```

**Production signal:** If you're writing `if isinstance(obj, SubClass)` to handle special cases, you're violating LSP.

---

### I — Interface Segregation Principle (ISP)
> **Don't force a class to implement interfaces it doesn't use**

Fat interfaces force implementors to have stub/empty methods — that's a design smell.

```python
# BAD — Fat interface
class NotificationService(ABC):
    @abstractmethod
    def send_email(self): ...
    @abstractmethod
    def send_sms(self): ...
    @abstractmethod
    def send_push(self): ...
    @abstractmethod
    def send_whatsapp(self): ...

class EmailOnlyNotifier(NotificationService):
    def send_email(self): ...
    def send_sms(self): raise NotImplementedError   # forced to stub!
    def send_push(self): raise NotImplementedError
    def send_whatsapp(self): raise NotImplementedError

# GOOD — Segregated interfaces
class EmailNotifier(ABC):
    def send_email(self): ...

class SMSNotifier(ABC):
    def send_sms(self): ...

class EmailOnlyNotifier(EmailNotifier):   # only implements what it needs
    def send_email(self): ...
```

---

### D — Dependency Inversion Principle (DIP)
> **Depend on abstractions, not concretions**

High-level modules (business logic) should not depend on low-level modules (DB, Redis, Kafka). Both should depend on interfaces.

```python
# BAD — OrderService is tightly coupled to MySQL
class OrderService:
    def __init__(self):
        self.db = MySQLDatabase()   # hard dependency

# GOOD — Depend on abstraction
class DatabaseInterface(ABC):
    @abstractmethod
    def save(self, data): ...

class MySQLDatabase(DatabaseInterface):
    def save(self, data): ...

class PostgresDatabase(DatabaseInterface):   # swap without touching OrderService
    def save(self, data): ...

class OrderService:
    def __init__(self, db: DatabaseInterface):   # inject dependency
        self.db = db
```

---

## 4. Production-Level Example

### Payment Processing System (Razorpay/PhonePe style)

```python
from abc import ABC, abstractmethod
from dataclasses import dataclass
from typing import List

@dataclass
class PaymentRequest:
    amount: float
    currency: str
    user_id: str
    method: str   # upi, card, netbanking

# SRP — Each class has one job
# DIP — All depend on abstractions

class PaymentGateway(ABC):
    @abstractmethod
    def process(self, request: PaymentRequest) -> dict: ...

class RazorpayGateway(PaymentGateway):
    def process(self, request: PaymentRequest) -> dict:
        # actual Razorpay API call
        return {"status": "success", "txn_id": "rzp_123"}

class StripeGateway(PaymentGateway):
    def process(self, request: PaymentRequest) -> dict:
        return {"status": "success", "txn_id": "stripe_456"}

# OCP — New gateway = new class, no change to PaymentService
class PaymentService:
    def __init__(self, gateway: PaymentGateway):   # DIP — injected
        self.gateway = gateway

    def pay(self, request: PaymentRequest) -> dict:
        return self.gateway.process(request)

# ISP — Separate fraud check interface
class FraudChecker(ABC):
    @abstractmethod
    def is_fraudulent(self, request: PaymentRequest) -> bool: ...

class MLFraudChecker(FraudChecker):
    def is_fraudulent(self, request: PaymentRequest) -> bool:
        return False   # calls ML model

# Usage — clean, testable, swappable
gateway = RazorpayGateway()
service = PaymentService(gateway)

# In tests — swap with mock, no real HTTP calls
class MockGateway(PaymentGateway):
    def process(self, request): return {"status": "success", "txn_id": "mock_001"}

test_service = PaymentService(MockGateway())
```

**Where SOLID appears in production at scale:**
- **SRP** → Each microservice owns one domain (Order, Payment, Notification)
- **OCP** → Plugin-based rule engines (add promo rules without touching core)
- **LSP** → Storage abstractions: S3, GCS, Azure Blob — all implement same `FileStorage` interface
- **ISP** → gRPC proto definitions — each service exposes only the methods it owns
- **DIP** → Dependency injection frameworks (Spring DI, Python FastAPI `Depends()`)

---

## 5. Internal Working

### How DIP powers microservices (step-by-step)

```
[OrderService] ──depends on──► [PaymentGatewayInterface]
                                        ▲
                          ┌─────────────┴──────────────┐
                   [RazorpayImpl]              [StripeImpl]
                          │                         │
                   [Razorpay API]            [Stripe API]
```

**Flow:**
1. `OrderService` is created with `PaymentGatewayInterface` injected by DI container
2. At runtime: DI container resolves → injects `RazorpayImpl`
3. `OrderService.process_payment()` calls `gateway.charge()` — doesn't know/care it's Razorpay
4. In staging env: DI container injects `MockGateway` — zero real API calls
5. When switching to Stripe: only change DI wiring config — `OrderService` untouched

**This is exactly how Spring Boot, FastAPI, and Go's interface system work.**

---

## 6. Most Common Interview Questions

---

### Q1: "Explain SOLID with a real example from your work"

**Strong answer:**
> "In my current project, we had a single `NotificationService` handling email, SMS, and push. Every new channel required modifying the class and re-testing all others. We refactored using SRP — split into `EmailService`, `SMSService`, `PushService` — and OCP — introduced a `Notifier` abstract base class. New channels are now added without touching existing ones. We also applied DIP by injecting the notifier through constructor, which made unit testing trivial with mocks."

**Follow-ups:**
- What tradeoffs did you face? (More classes, but each is simpler)
- How did you handle backward compatibility?

**Common mistake:** Giving a textbook definition instead of a story from experience.

---

### Q2: "What's the difference between SRP and Separation of Concerns?"

**Strong answer:**
> "SRP is a class-level principle — one class, one reason to change. Separation of Concerns is a broader architectural principle — different layers (controller, service, repository) handle different concerns. SRP is one way to implement SoC at the class level. In practice, they're complementary — SoC defines layers, SRP keeps each layer's classes focused."

**Follow-up:** How does this apply to microservices? (Each service = one business domain = SoC at service level)

---

### Q3: "When would you violate SOLID deliberately?"

**Strong answer:**
> "In scripts, CLIs, data migrations — applying SOLID to a 50-line one-time script is over-engineering. Also in very early-stage startups where speed matters. SOLID has a cost: more classes, more files, more indirection. You apply it when the codebase will live long enough to justify that cost — typically when you're building a product that'll evolve over years."

**Follow-up:** How do you know when to refactor? (When change becomes painful — that's the signal)

---

### Q4: "How does DIP relate to Dependency Injection?"

**Strong answer:**
> "DIP is the principle — depend on abstractions. Dependency Injection is a pattern that implements it. DI means: don't create your dependencies inside the class, receive them from outside. FastAPI's `Depends()`, Spring's `@Autowired`, and constructor injection in Go are all DI patterns that enforce DIP. DIP tells you what to depend on, DI tells you how to wire it."

---

### Q5: "Explain LSP with a violation example"

**Strong answer:**
> "Classic example: If you have a `Bird` class with a `fly()` method, and you create a `Penguin` subclass that throws an exception in `fly()` — that's an LSP violation. Code that iterates a list of `Bird` objects and calls `fly()` will blow up. The fix is to not force `Penguin` into a `FlyingBird` hierarchy. Separate `FlyingBird` and `NonFlyingBird` or remove `fly()` from the base."

---

## 7. Senior-Level Understanding

### Tradeoffs

| Principle | Benefit | Cost |
|---|---|---|
| SRP | Easy to test, change | More classes, more navigation |
| OCP | Safe extension | Requires upfront abstraction — premature abstraction is dangerous |
| LSP | Reliable polymorphism | Forces careful hierarchy design |
| ISP | Lean interfaces | More interfaces to manage |
| DIP | Testability, swappability | DI containers add complexity |

### When NOT to use SOLID

- **Simple scripts or CLIs** — No long-term maintenance, SOLID adds friction
- **Data pipelines / ETL** — Procedural often cleaner than over-abstracted OOP
- **Premature abstraction** — Don't create interfaces before you have 2 concrete implementations. "Rule of Three" — abstract after the third time
- **Small teams, early startup** — Velocity > correctness in week 1

### Performance implications
- **DIP** introduces a layer of indirection (virtual dispatch) — negligible at application layer, but avoid in tight loops
- **ISP** — Many small interfaces don't hurt performance; it's a design-time concern
- **SRP** — More objects can mean more heap allocations in memory-constrained systems (rare concern at app server level)

### Scalability implications
- **DIP + SRP** are the real scalability enablers — they make it possible to split monolith into microservices cleanly along responsibility boundaries
- **OCP** enables feature flags, A/B testing patterns — new behavior as new class, toggled by config

---

## 8. Comparison Section

### Composition vs Inheritance (directly tied to SOLID — LSP, OCP)

| Aspect | Inheritance | Composition |
|---|---|---|
| Coupling | Tight — subclass depends on parent internals | Loose — depends on interface |
| LSP risk | High — subclass can break contract | Low — compose only needed behavior |
| Flexibility | Low — hierarchy fixed at compile time | High — swap behaviors at runtime |
| Testability | Harder — must mock entire parent | Easy — mock each composed piece |
| When to use | True IS-A relationship | HAS-A or CAN-DO relationship |
| Production example | `Exception` hierarchies | Strategy pattern, Payment gateways |

**Rule of thumb:** "Favor composition over inheritance" — this is a senior-level signal in interviews.

---

### SOLID vs YAGNI vs DRY

| Principle | Focus | Watch out for |
|---|---|---|
| SOLID | Good structure | Over-engineering |
| DRY (Don't Repeat Yourself) | Reduce duplication | Wrong abstractions — wrong DRY is worse than duplication |
| YAGNI (You Aren't Gonna Need It) | Don't build what you don't need | Conflict with OCP — balance both |

**Senior insight:** SOLID and YAGNI are in tension. SOLID says "plan for extension," YAGNI says "don't plan ahead." Answer: apply SOLID where change is _predictable_ (payment methods, notification channels), apply YAGNI where it's speculative.

---

## 9. Production Debugging / Failure Cases

### Failure 1: OCP Violation — The Promo Code Nightmare
**Scenario:** At an e-commerce company, `PriceCalculator` had 15 `if-elif` blocks for discount types. During Diwali sale, adding a new discount caused a regression in the "bulk discount" path.

**Debug:** Git blame showed 8 engineers modified the same class in 3 months. Classic OCP violation.

**Fix:** Strategy pattern — each discount type in its own class, registered in a discount registry. New discount = new class + one registry entry.

**Monitoring signal:** If one class has commits from 5+ engineers in a sprint, SRP/OCP is likely broken.

---

### Failure 2: DIP Violation — Untestable Payment Module
**Scenario:** `PaymentService` directly instantiated `RazorpayClient`. Unit tests made real HTTP calls. CI pipeline started failing intermittently due to Razorpay sandbox downtime.

**Debug:** Coverage report showed 0% on payment module — engineers gave up writing tests.

**Fix:** Extract `PaymentGateway` interface. Inject via constructor. Tests use `MockGateway` — zero HTTP calls.

**Monitoring:** If your CI test suite requires external services (DB, payment APIs) to be up — you have DIP violations.

---

### Failure 3: LSP Violation — Silent Data Corruption
**Scenario:** `ReadOnlyCache` extended `Cache` but silently ignored `write()` calls (returned `True` without actually writing). Code that checked `if cache.write(key, val)` assumed data was cached — leading to stale data in production.

**Debug:** Took 3 days to find — no exception, just silent no-op.

**Fix:** `ReadOnlyCache` should not extend `Cache` if it can't honor the write contract. Separate `ReadableCache` and `WritableCache` interfaces.

**Rule:** Violations of LSP often produce silent failures — hardest to debug.

---

## 10. Important Code Examples

### FastAPI + DIP (Production-style)

```python
from abc import ABC, abstractmethod
from fastapi import FastAPI, Depends

# Abstraction layer
class UserRepository(ABC):
    @abstractmethod
    def get_user(self, user_id: str) -> dict: ...

    @abstractmethod
    def save_user(self, user: dict) -> bool: ...

# Concrete implementations
class PostgresUserRepository(UserRepository):
    def __init__(self, db_conn):
        self.conn = db_conn

    def get_user(self, user_id: str) -> dict:
        # real DB query
        return self.conn.execute("SELECT * FROM users WHERE id = %s", user_id)

    def save_user(self, user: dict) -> bool:
        self.conn.execute("INSERT INTO users ...", user)
        return True

class InMemoryUserRepository(UserRepository):   # for tests
    def __init__(self):
        self.store = {}

    def get_user(self, user_id: str) -> dict:
        return self.store.get(user_id)

    def save_user(self, user: dict) -> bool:
        self.store[user["id"]] = user
        return True

# Service layer — SRP: only business logic
class UserService:
    def __init__(self, repo: UserRepository):   # DIP
        self.repo = repo

    def get_profile(self, user_id: str) -> dict:
        user = self.repo.get_user(user_id)
        if not user:
            raise ValueError("User not found")
        return user

# FastAPI DI wiring
app = FastAPI()

def get_user_repo() -> UserRepository:
    db = get_db_connection()         # from connection pool
    return PostgresUserRepository(db)

@app.get("/users/{user_id}")
def get_user(user_id: str, repo: UserRepository = Depends(get_user_repo)):
    service = UserService(repo)
    return service.get_profile(user_id)
```

---

### OCP with Strategy Pattern — Notification System

```python
from abc import ABC, abstractmethod
from typing import List

class NotificationChannel(ABC):
    @abstractmethod
    def send(self, user_id: str, message: str) -> bool: ...

class EmailChannel(NotificationChannel):
    def send(self, user_id: str, message: str) -> bool:
        # call SES/SendGrid
        print(f"Email sent to {user_id}: {message}")
        return True

class SMSChannel(NotificationChannel):
    def send(self, user_id: str, message: str) -> bool:
        # call Twilio
        print(f"SMS sent to {user_id}: {message}")
        return True

class WhatsAppChannel(NotificationChannel):    # NEW: only adding, not modifying
    def send(self, user_id: str, message: str) -> bool:
        print(f"WhatsApp sent to {user_id}: {message}")
        return True

# NotificationService is CLOSED for modification, OPEN for extension
class NotificationService:
    def __init__(self, channels: List[NotificationChannel]):
        self.channels = channels

    def notify(self, user_id: str, message: str):
        for channel in self.channels:
            try:
                channel.send(user_id, message)
            except Exception as e:
                # log and continue — one channel failure shouldn't block others
                print(f"Channel {channel.__class__.__name__} failed: {e}")

# Wiring — driven by config or feature flags
channels = [EmailChannel(), SMSChannel()]
# Adding WhatsApp: just add to the list — no existing class touched
# channels = [EmailChannel(), SMSChannel(), WhatsAppChannel()]
service = NotificationService(channels)
```

---

## 11. Revision Notes — Must-Know Interview Priority

```
╔══════════════════════════════════════════════════════════════════╗
║               SOLID — QUICK REVISION CHEATSHEET                 ║
╠══════════════════════════════════════════════════════════════════╣
║  S — SRP   One class = One reason to change                     ║
║            Signal: class with 5+ unrelated methods              ║
║                                                                  ║
║  O — OCP   New feature = new class, not modified existing       ║
║            Pattern: Strategy, Plugin, Registry                  ║
║                                                                  ║
║  L — LSP   Subclass must fully honor parent's contract          ║
║            Signal: isinstance() checks, NotImplementedError     ║
║                                                                  ║
║  I — ISP   Don't force empty/stub method implementations        ║
║            Signal: raise NotImplementedError in subclass        ║
║                                                                  ║
║  D — DIP   Depend on interfaces, not concrete classes           ║
║            Pattern: Constructor Injection, Depends()            ║
╠══════════════════════════════════════════════════════════════════╣
║  MOST ASKED in interviews: SRP, OCP, DIP                        ║
║  HIGHEST production value: DIP (testability), OCP (extensible)  ║
║  SNEAKY violations: LSP (silent failures), ISP (fat interfaces) ║
╠══════════════════════════════════════════════════════════════════╣
║  KEY PATTERNS THAT IMPLEMENT SOLID                               ║
║  Strategy Pattern    → OCP                                       ║
║  Repository Pattern  → DIP + SRP                                 ║
║  Factory Pattern     → DIP                                       ║
║  Observer Pattern    → OCP + SRP                                 ║
╠══════════════════════════════════════════════════════════════════╣
║  SENIOR SIGNAL answers:                                          ║
║  "SOLID has cost — more classes, more indirection"               ║
║  "Apply where change is predictable, not everywhere"             ║
║  "DIP makes services testable without real infrastructure"       ║
║  "Favor composition over inheritance — OCP + LSP safety"         ║
╚══════════════════════════════════════════════════════════════════╝
```

---

*Remember: In interviews, connect every principle to testability, extensibility, or a production failure you've seen or can reason about. That's what separates 20 LPA answers from 10 LPA answers.*
